# 7.16 舊例子混回去有用嗎？


只練加法會改變屬性回答的共享參數，可以在新batch再放少量舊屬性題，讓兩種目標都給出更新訊號。這叫回放：用當前參數重新算舊資料，不是播放舊梯度。

例如舊題「物體紅色、圓形；請回形狀」應答「圓形」，新題「1+1=?」應答「2」。以下A與B各兩筆、每筆假設兩個有效答案，合成四筆八答案。source记錄来源；文字只用「舊題一」等代號表示這份配方，真訓練要換完整問答。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
a = [
    {"source": "A", "text": "舊題一", "answer_tokens": 2},
    {"source": "A", "text": "舊題二", "answer_tokens": 2},
]
b = [
    {"source": "B", "text": "新題一", "answer_tokens": 2},
    {"source": "B", "text": "新題二", "answer_tokens": 2},
]
recipe = a + b
print("每批來源", [record["source"] for record in recipe])
print("有效答案預算", sum(record["answer_tokens"] for record in recipe))
print("待測B-only與replay各自的A/B成績")

輸入兩筆A與兩筆B，每筆假設兩個有效答案token，合成四筆batch，所以來源A、A、B、B，token總數8，兩邊各半。text是來源示意名，不是完整訓練對話；真正數據應使用實際問答與測得答案長度。這段只計配方與預算，未訓練，不輸出虛構保留能力曲線。

固定batch四筆時，B-only可用四筆B共八token，replay用兩筆A兩筆B同樣八token；總預算一致，但replay新任務資料份額少一些。這是方法真實的取捨，不能一邊讓B兩筆再加A兩筆額外跑一輪、另一邊只給B原兩筆，然後說都是同樣成本。

同batch四筆時，B-only四筆B和兩A兩B都八答案；回放卻分走一些新任務份額，這是實際取捨。已有回放支线保留舊題6/10，B-only為0/10，新題兩邊仍0/7；舊答案较長也使回放有效目標接近翻倍，不能說已隔離同token成本下的純效果。

練習把recipe改成 `[a[0],b[0],b[1],b[0]]`，先預測來源25:75、有效token仍8，再執行核對。這份小練習為保持四筆而重複一B例，實際實驗應換不同B資料；它不增加資料覆蓋。若把B答案長度改4，重算token比例後就不再是25:75，這能把回放配方接回本章的有效監督統計。

<details>
<summary>補充：實作約定與原始紀錄</summary>

把[7.15的實跑](https://birdhackor.github.io/tiny-perceptron-vlm/7.15.html)加上回放支線，兩支都從同一份屬性基模開始、更新500次，每批16題、學習率0.003。B-only只抽49筆加法訓練題；replay則從45筆舊屬性題加49筆加法題一起抽。下面比較完整答案ID，兩個任務與生成上限始終相同：

| 模型狀態 | 舊A驗證／最後檢查答對 | 新B驗證／最後檢查答對 | 這段訓練的有效目標 |
| --- | --- | --- | ---: |
| 尚未繼續訓練 | 1／5、5／10 | 0／8、0／7 | — |
| 只練B，500次 | 0／5、0／10 | 0／8、0／7 | 18,453 |
| 混回A，500次 | 4／5、6／10 | 0／8、0／7 | 36,384 |

回放支線保住了這批舊題的表現，最後檢查還多答對一題；但是B兩支都沒有通過留出加法，所以不能說回放已讓兩種能力一起學好。兩支與起點在全部這些題都會生成EOS，內容成績的差異並非漏掉結束標記。A的驗證／最後檢查分母是36／69個有效答案目標，B是16／14個，各支完整計分，沒有挑掉錯題。

還要讀右欄：相同步數與筆數，不代表相同的監督總量。舊屬性答案比加法數字長，混回A後這次有效目標接近翻倍；輸入也較長，運算成本不能視為已完全匹配。這個實驗支持「此配方下舊任務保留較好」，沒有隔離成同token預算下的純回放效應。[完整報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/sft_ablation.json)保留各支訓練與留出生成，重跑方式見[T.4](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
